In [4]:
import pandas as pd
import numpy as np
import os

INPUT="summary.csv"

OUTDIR="tables"

os.makedirs(OUTDIR,exist_ok=True)

df=pd.read_csv(INPUT)

OUTLIERS=["ZScore","IsolationForest","IQR"]

MODELS=sorted(df["model"].unique())

def fmt(x):

    if pd.isna(x):

        return "NA"

    return f"{x:.4f}"

def fmt_delta(x):

    if pd.isna(x):

        return "NA"

    sign="+" if x>=0 else ""

    return f"{sign}{x:.4f}"

for model in MODELS:

    mdf=df[df["model"]==model]

    datasets=sorted(mdf["dataset"].unique())

    rows=[]

    deltas={o:[] for o in OUTLIERS}

    for d in datasets:

        sub=mdf[mdf["dataset"]==d]

        base=sub[sub["outlier"].isna()]

        if len(base)==0:

            continue

        base_mean=base.iloc[0]["mean"]

        row={"Dataset":d}

        row["None"]=fmt(base_mean)

        for o in OUTLIERS:

            filt=sub[sub["outlier"]==o]

            if len(filt)==0:

                row[o]="NA"

                row[f"Δ {o}"]="NA"

                continue

            m=filt.iloc[0]["mean"]

            delta=m-base_mean

            row[o]=fmt(m)

            row[f"Δ {o}"]=fmt_delta(delta)

            deltas[o].append(delta)

        rows.append(row)

    # average delta row
    avg_row={"Dataset":"AVG Δ"}

    avg_row["None"]=""

    for o in OUTLIERS:

        avg_row[o]=""

        if len(deltas[o])>0:

            avg_row[f"Δ {o}"]=fmt_delta(

                np.mean(deltas[o])

            )

        else:

            avg_row[f"Δ {o}"]="NA"

    rows.append(avg_row)

    table=pd.DataFrame(rows)

    table=table.sort_values("Dataset")

    table.to_csv(

        f"{OUTDIR}/{model}_delta_table.csv",

        index=False
    )

print("Delta tables created")

Delta tables created
